# 01 — Data Collection

Fetches all raw data for the Austria Energy & Climate project.

| Source | What we get | Credentials needed |
|---|---|---|
| Open-Meteo / ERA5 | Hourly weather (Vienna) | None |
| Our World in Data | Annual energy & CO₂ time series | None |
| ENTSO-E | Hourly generation, load, prices | API key (free) |

**Run order:** cells 1–4 work immediately. Cell 5 (ENTSO-E) requires your API key in `.env`.

## 0 · Setup

In [ ]:
# ── install dependencies if needed ────────────────────────────────────────────
# Run this cell once, then restart the kernel.
# !pip install entsoe-py requests pandas python-dotenv

In [ ]:
import sys
from pathlib import Path

# make src/ importable from the notebook
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# load .env so ENTSOE_API_KEY is available as an env variable
from dotenv import load_dotenv  # noqa: E402
load_dotenv(PROJECT_ROOT / ".env")  # pyright: ignore[reportUnusedCallResult]

from src.data_loader import DataLoader  # noqa: E402

print("Project root:", PROJECT_ROOT)

## 1 · Our World in Data — annual energy series

Downloads the full OWID energy CSV (~10 MB) and saves the Austria slice.  
No credentials required.

In [ ]:
import pandas as pd

dl = DataLoader()   # no API key needed for OWID + weather
owid_path = dl.fetch_owid()

df_owid = pd.read_csv(owid_path, index_col="year")
print(f"Shape: {df_owid.shape}")
df_owid.tail()

In [ ]:
# Quick sanity check: which columns do we have?
energy_cols = [c for c in df_owid.columns if any(
    kw in c for kw in ["electricity", "renewables", "carbon", "energy", "co2", "solar", "wind", "hydro"]
)]
print(f"{len(energy_cols)} energy-related columns:")
for c in energy_cols:
    print(" ", c)

## 2 · Open-Meteo — hourly weather (Vienna)

ERA5 reanalysis data via the free Open-Meteo API.  
No account or API key needed.

In [ ]:
START = "2019-01-01"
END   = "2024-12-31"

weather_path = dl.fetch_weather(start=START, end=END)

df_weather = pd.read_csv(weather_path, index_col="timestamp", parse_dates=True)
print(f"Shape: {df_weather.shape}")
print(f"Date range: {df_weather.index.min()} → {df_weather.index.max()}")
df_weather.head()

In [ ]:
# Basic stats + missing value check
print("=== Missing values ===")
print(df_weather.isna().sum())
print()
print("=== Descriptive stats ===")
df_weather.describe().round(2)

In [ ]:
# Quick visual: daily mean temperature to confirm data looks right
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

fig, (ax_temp, ax_solar) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

daily = df_weather.resample("D").mean()

ax_temp.plot(daily.index, daily["temperature_2m"], lw=0.8, color="#E8593C")
ax_temp.set_ylabel("Temperature (°C)")
ax_temp.set_title("Vienna — daily mean temperature & solar radiation (ERA5)")

ax_solar.plot(daily.index, daily["shortwave_radiation"], lw=0.8, color="#EF9F27")
ax_solar.set_ylabel("Solar radiation (W/m²)")

for ax in (ax_temp, ax_solar):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "data" / "raw" / "weather_sanity_check.png", dpi=120)
plt.show()
print("Looks reasonable? Seasonal oscillations present -> ok")

## 3 · ENTSO-E — hourly generation, load & prices

**Pre-requisite:** Add your API key to `.env`:
```
ENTSOE_API_KEY=your-key-here
```
Then re-run cell 0 (`load_dotenv`) and continue from here.

If the key is not set yet, this section will log a warning and skip — that's fine.

In [ ]:
# Re-initialise with API key now that .env is loaded
dl_entsoe = DataLoader()   # picks up ENTSOE_API_KEY from env automatically

if dl_entsoe._entsoe_client is None:
    print("⚠️  ENTSO-E client not available — add your key to .env and re-run.")
else:
    print("✓  ENTSO-E client ready.")

In [ ]:
# ── Fetch generation ──────────────────────────────────────────────────────────
# Takes ~2–4 minutes for 2019–2024 (6 API calls, one per year).
# Safe to interrupt and resume — each year chunk is independent.

if dl_entsoe._entsoe_client:
    ts_start = pd.Timestamp(START, tz="UTC")
    ts_end   = pd.Timestamp(END,   tz="UTC") + pd.Timedelta(days=1)

    gen_path = dl_entsoe.fetch_generation(ts_start, ts_end)
    assert gen_path is not None
    df_gen = pd.read_csv(
    gen_path,
    header=[0, 1],
    index_col=0,
    parse_dates=True,
    )
    df_gen.index = pd.to_datetime(df_gen.index, utc=True)
    df_gen = df_gen.sort_index()
    print(f"Shape: {df_gen.shape}")
    print("Columns (fuel types):", df_gen.columns.tolist())
    df_gen.head()

In [ ]:
df_gen.index = pd.to_datetime(df_gen.index, utc=True)
df_gen = df_gen.sort_index()
print(f"Shape: {df_gen.shape}")
print("Columns (fuel types):", df_gen.columns.tolist())
df_gen.head()

In [ ]:
# ── Fetch load ────────────────────────────────────────────────────────────────
if dl_entsoe._entsoe_client:
    load_path = dl_entsoe.fetch_load(ts_start, ts_end)
    assert load_path is not None
    df_load = pd.read_csv(load_path, index_col=0, parse_dates=True)
    df_load.index = pd.to_datetime(df_load.index, utc=True)
    df_load = df_load.sort_index()
    print(f"Shape: {df_load.shape}")
    df_load.head()

In [ ]:
# ── Fetch day-ahead prices ────────────────────────────────────────────────────
if dl_entsoe._entsoe_client:
    prices_path = dl_entsoe.fetch_prices(ts_start, ts_end)
    assert prices_path is not None
    df_prices = pd.read_csv(prices_path, index_col=0, parse_dates=True)
    #* Added this line to handel DST problem
    # TODO: Check with Claude
    
    df_prices.index = pd.to_datetime(df_prices.index, utc=True)
    df_prices = df_prices.sort_index()
    print(f"Shape: {df_prices.shape}")
    df_prices.head()

In [ ]:
# ── Sanity check: plot 1 week of generation mix ───────────────────────────────
if dl_entsoe._entsoe_client:
    week = df_gen.xs("Actual Aggregated", axis=1, level=1).loc["2023-07-01":"2023-07-07"]
    #* Changed this line to handell two level columns
    # TODO: Check with Claude, because later it tried to get pass that
    
    # Keep only columns with actual data
    week = week.dropna(axis=1, how="all")
    # Remove "Actual Aggregated" or "Actual Consumption" meta-columns if present
    fuel_cols = [c for c in week.columns if "Actual" not in str(c)]
    week = week[fuel_cols]

    fig, ax = plt.subplots(figsize=(14, 5))
    week.plot.area(ax=ax, stacked=True, linewidth=0)
    ax.set_title("Austria — hourly generation mix (sample week, July 2023)")
    ax.set_ylabel("MW")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "data" / "raw" / "generation_sanity_check.png", dpi=120)
    plt.show()

In [ ]:
#* Wrote it with cursor, to understand it
# TODO: Check it later


# ── Pumped storage: net flow (generation − consumption) ──────────────────────
# Positive = sending power to the grid (turbines)
# Negative = drawing power from the grid (pumping water uphill)
if dl_entsoe._entsoe_client:
    gen  = df_gen[("Hydro Pumped Storage", "Actual Aggregated")]
    cons = df_gen[("Hydro Pumped Storage", "Actual Consumption")]
    net  = (gen - cons).loc["2023-07-01":"2023-07-07"]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(net.index, net.values, 0,  # pyright: ignore[reportArgumentType]
                    where=(net.values >= 0), color="#2ca02c", alpha=0.7,  # pyright: ignore[reportOperatorIssue, reportArgumentType]
                    label="Generating (turbine)")
    ax.fill_between(net.index, net.values, 0,  # pyright: ignore[reportArgumentType]
                    where=(net.values < 0),  color="#d62728", alpha=0.7,  # pyright: ignore[reportOperatorIssue, reportArgumentType]
                    label="Pumping (consuming)")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Austria — Hydro Pumped Storage net flow (sample week, July 2023)")
    ax.set_ylabel("MW   (+ generating  /  − pumping)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()

## 4 · Summary — what we have

Run this cell after all fetches to confirm everything landed.

In [ ]:
raw_dir = PROJECT_ROOT / "data" / "raw"
ext_dir = PROJECT_ROOT / "data" / "external"

print("=== data/raw/ ===")
for f in sorted(raw_dir.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    df_tmp  = pd.read_csv(f, nrows=1)
    print(f"  {f.name:<40}  {size_kb:>8.1f} KB")

print()
print("=== data/external/ ===")
for f in sorted(ext_dir.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40}  {size_kb:>8.1f} KB")

print()
print("Phase 1 complete ✓")
print("Next: notebook 02_cleaning_eda.ipynb")